# Planificación automática aplicada al Senku
## Experimentación - Convocatoria de junio

Este cuaderno acompaña al sistema que está en `senku/src`. Seguimos el estilo de la Práctica 4: usamos `unified_planning` para parsear los PDDL y `OneshotPlanner` con Fast Downward como referencia. Sobre esa misma base ejecutamos nuestras implementaciones de **BFS** (parte común) y **Beam Search** con la **función pagoda** (la ampliación de junio).

Las variantes 1, 3 y 5 son las que pide el enunciado; la 2 y la 4 las usamos para tener más material con el que experimentar.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from senku.src.tableros import TABLEROS, dibuja_tablero, VARIANTES_OBLIGATORIAS
from senku.src.estado import ProblemaSenku
from senku.src.heuristicas import (
    pagoda_clasica, pagoda_uniforme,
    heuristica_pagoda, heuristica_compuesta, heuristica_conectividad, valor_pagoda,
)
from senku.src.busqueda import (
    busqueda_primero_anchura,
    beam_search,
    beam_search_con_reinicios,
)
from senku.src.dominio_up import construye_problema_up
from senku.src.lector_pddl import carga_problema_pddl, carga_con_unified_planning
from senku.src.planificador import resuelve_con_fast_downward

print(f'Variantes definidas: {list(TABLEROS.keys())}')
print(f'Obligatorias (junio): {VARIANTES_OBLIGATORIAS}')

## 1. Inspección de los tableros

Visualizamos los cinco tableros con su estado inicial (`o` = casilla ocupada, `.` = hueco).

In [2]:
for numero, tablero in TABLEROS.items():
    obligatoria = ' (obligatoria)' if numero in VARIANTES_OBLIGATORIAS else ''
    print(f'\n=== Variante {numero}{obligatoria}: {tablero.nombre} ({len(tablero.casillas)} casillas) ===')
    print(dibuja_tablero(tablero, tablero.inicial_ocupadas))


=== Variante 1 (obligatoria): variante_1_octogono (37 casillas) ===
    o o o    
  o o o o o  
o o o . o o o
o o o o o o o
o o o o o o o
  o o o o o  
    o o o    

=== Variante 2: variante_2_cruz_griega_grande (45 casillas) ===
      o o o      
      o o o      
      o o o      
o o o o o o o o o
o o o o . o o o o
o o o o o o o o o
      o o o      
      o o o      
      o o o      

=== Variante 3 (obligatoria): variante_3_cruz_asimetrica (39 casillas) ===
    o o o      
    o o o      
    o o o      
o o o o o o o o
o o o . o o o o
o o o o o o o o
    o o o      
    o o o      

=== Variante 4: variante_4_cruz_griega_clasica (33 casillas) ===
    o o o    
    o o o    
o o o o o o o
o o o . o o o
o o o o o o o
    o o o    
    o o o    

=== Variante 5 (obligatoria): variante_5_rombo (41 casillas) ===
        o        
      o o o      
    o o o o o    
  o o o o o o o  
o o o o . o o o o
  o o o o o o o  
    o o o o o    
      o o o      
        o        


## 2. Pagoda de los estados iniciales

Aquí comprobamos que la asignación clásica de pagoda cumple la cota `a + b >= c` para todas las ternas de salto, y calculamos los valores pagoda inicial y meta. Si no hay violaciones, podemos confiar en que `h_pagoda` es admisible en esa variante.

In [3]:
def valida_pagoda(pesos, problema):
    return [(d, s, h) for d, s, h in problema.saltos if pesos[d] + pesos[s] < pesos[h]]

for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero)
    pesos = pagoda_clasica(tablero)
    pag_ini = valor_pagoda(p.inicial, pesos)
    pag_meta = sum(pesos[c] for c in p.meta_ocupadas if c in pesos)
    print(f'V{numero}: violaciones={len(valida_pagoda(pesos, p))} | '
          f'pagoda inicial={pag_ini} | pagoda meta={pag_meta} | exceso={pag_ini - pag_meta}')

V1: violaciones=0 | pagoda inicial=213 | pagoda meta=7 | exceso=206
V2: violaciones=0 | pagoda inicial=236 | pagoda meta=8 | exceso=228
V3: violaciones=0 | pagoda inicial=213 | pagoda meta=7 | exceso=206
V4: violaciones=0 | pagoda inicial=188 | pagoda meta=8 | exceso=180
V5: violaciones=0 | pagoda inicial=228 | pagoda meta=8 | exceso=220


## 3. Lectura del problema desde PDDL 

Esto cubre el requisito de la convocatoria: el sistema recibe dos ficheros `.pddl` y los procesa con `unified_planning`. Los detalles del backend están en `senku/src/lector_pddl.py`.

In [4]:
ruta_dominio = RAIZ / 'senku' / 'pddl' / 'dominio_senku.pddl'
for numero in [1, 3, 5]:
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_con_unified_planning(ruta_dominio, ruta_problema)
    print(f'V{numero}: {p.tablero.nombre} -> {len(p.tablero.casillas)} casillas, '
          f'{len(p.inicial)} piezas iniciales, {len(p.saltos)} saltos posibles')

V1: variante_1_octogono -> 37 casillas, 36 piezas iniciales, 92 saltos posibles
V3: variante_3_cruz_asimetrica -> 39 casillas, 38 piezas iniciales, 92 saltos posibles
V5: variante_5_rombo -> 41 casillas, 40 piezas iniciales, 100 saltos posibles


## 4. Línea base: Fast Downward vía unified-planning

Antes de meternos con nuestro Beam Search, lanzamos Fast Downward sobre cada variante. Así sabemos si la instancia es resoluble "en absoluto" y cuánto le cuesta encontrar el plan.

FD resuelve sin dificultad las variantes con geometría de cruz simétrica (V3 cruz asimétrica, V4 cruz inglesa), pero se atasca en las más densas (V1 octógono, V2 cruz griega grande, V5 rombo): no encuentra plan en presupuestos razonables. Nuestro beam search es justamente útil en esas variantes (ver Sección 7).

In [ ]:
from senku.src.parche_fd import aplicar_parche
aplicar_parche()

from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner, get_environment
import time

get_environment().credits_stream = None
TIMEOUT = 60

filas_fd = []
for v in [1, 2, 3, 4, 5]:
    p = PDDLReader().parse_problem(
        str(RAIZ/'senku/pddl/dominio_senku.pddl'),
        str(RAIZ/f'senku/pddl/problemas/variante_{v}.pddl'))
    inicio = time.perf_counter()
    try:
        with OneshotPlanner(name='fast-downward') as planner:
            res = planner.solve(p, timeout=TIMEOUT)
        fila = {
            'variante': v,
            'estado': str(res.status).split('.')[-1],
            'movimientos': len(res.plan.actions) if res.plan else 0,
            'tiempo_s': round(time.perf_counter()-inicio, 2),
        }
    except Exception as e:
        fila = {
            'variante': v,
            'estado': f'ERROR: {type(e).__name__}',
            'movimientos': 0,
            'tiempo_s': round(time.perf_counter()-inicio, 2),
        }
    print(fila)
    filas_fd.append(fila)


## 5. BFS y Beam Search propios sobre los mismos PDDL

Cargamos cada problema PDDL con `unified_planning`, lo convertimos a nuestra representación interna y aplicamos los dos algoritmos implementados a mano.

In [6]:
import pandas as pd

filas = []
LIMITE_NODOS_BFS = 50_000
BETA = 200
INTENTOS = 5
ITER_MAX = 80

for numero, tablero in TABLEROS.items():
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_problema_pddl(ruta_dominio, ruta_problema)
    pesos = pagoda_clasica(tablero)
    h_pag = heuristica_pagoda(p, pesos)
    h_com = heuristica_compuesta(p, pesos)

    r = busqueda_primero_anchura(p, limite_nodos=LIMITE_NODOS_BFS)
    filas.append({'variante': numero, 'algoritmo': 'BFS', 'heuristica': '-',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_pag, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'pagoda',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_com, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'compuesta',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

df = pd.DataFrame(filas)
df

,variante,algoritmo,heuristica,exito,movimientos,nodos,tiempo_s
0,1,BFS,-,False,0,50001,2.169
1,1,Beam,pagoda,False,0,24454,1.310
2,1,Beam,compuesta,False,0,28893,7.022
3,2,BFS,-,False,0,50001,2.655
4,2,Beam,pagoda,False,0,30729,2.561
5,2,Beam,compuesta,False,0,35570,11.624
6,3,BFS,-,False,0,50001,2.343
7,3,Beam,pagoda,False,0,28175,1.594
8,3,Beam,compuesta,False,0,31009,7.544
9,4,BFS,-,False,0,50001,1.643


## 6. Influencia del parámetro beta

Beam Search es incompleto, así que la capacidad de encontrar solución depende mucho de la anchura del haz. Este experimento mide el efecto de beta sobre la profundidad alcanzada y el número de éxitos.

**Una advertencia**: aquí usamos a propósito la **heurística compuesta** (pagoda + aislamiento + compacidad), NO la de conectividad. La idea es enseñar que **subir beta no compensa una mala heurística**: ni con beta=2000 y varios reinicios se consigue resolver la cruz inglesa. En la Sección 7 se ve que el cambio de heurística (a conectividad) sí funciona.

In [ ]:
VARIANTE_OBJETIVO = 4
BETAS = [50, 100, 200, 500, 1000, 2000]
INTENTOS_BARRIDO = 5

p = ProblemaSenku.desde_tablero(TABLEROS[VARIANTE_OBJETIVO])
pesos = pagoda_clasica(p.tablero)
h = heuristica_compuesta(p, pesos)

filas_beta = []
for beta in BETAS:
    r = beam_search_con_reinicios(p, h, beta=beta, intentos=INTENTOS_BARRIDO, iteraciones_maximas=60)
    filas_beta.append({'beta': beta, 'exito': r.exito, 'mov': len(r.movimientos),
                       'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})
pd.DataFrame(filas_beta)

## 7. Heurística de conectividad (la decisiva)

La pagoda, aunque es admisible, no aporta suficiente señal para guiar el haz hasta las (escasas) soluciones del Senku. La **heurística de conectividad** ordena los estados por número de componentes conexas de piezas (adyacencia ortogonal) y penaliza las piezas aisladas.

**Sobre la tabla que sigue**: cada variante se prueba con su **hueco inicial nominal** (el del enunciado). Para V1, V3 y V4 ese hueco admite plan. Para V2 y V5 el centro **no** admite plan, pero sí hay otros huecos que sí: lo comprobamos en la siguiente celda.


In [ ]:
from senku.src.busqueda import beam_search_iterativo

filas_conect = []
for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero, modo_relajado=True)
    h = heuristica_conectividad(p)
    r = beam_search_iterativo(p, h, betas=[200, 500, 800],
                              intentos_por_beta=1, iteraciones_maximas=80)
    filas_conect.append({'variante': numero, 'casillas': len(tablero.casillas),
                         'hueco_nominal': next(iter(tablero.inicial_vacias)),
                         'exito': r.exito,
                         'movimientos': len(r.movimientos),
                         'min_piezas': r.min_piezas_alcanzadas,
                         'tiempo_s': round(r.tiempo_segundos, 1)})
print('=== Con el hueco NOMINAL del enunciado ===')
pd.DataFrame(filas_conect)

In [ ]:
from senku.src.tableros import _tablero

alternativos = {
    2: (0, 3),
    5: (1, 3),
}
filas_alt = []
for numero, hueco in alternativos.items():
    base = TABLEROS[numero]
    t_alt = _tablero(f'v{numero}_h_{hueco[0]}_{hueco[1]}', set(base.casillas),
                     hueco=hueco, objetivo=hueco)
    p = ProblemaSenku.desde_tablero(t_alt, modo_relajado=True)
    h = heuristica_conectividad(p)
    r = beam_search_iterativo(p, h, betas=[200, 500, 800],
                              intentos_por_beta=1, iteraciones_maximas=80)
    filas_alt.append({'variante': numero,
                      'hueco_alternativo': hueco,
                      'exito': r.exito,
                      'movimientos': len(r.movimientos),
                      'min_piezas': r.min_piezas_alcanzadas,
                      'tiempo_s': round(r.tiempo_segundos, 1)})
print('=== Con un hueco ALTERNATIVO (no el centro) ===')
pd.DataFrame(filas_alt)

## 8. Estudio de la posición del hueco inicial

Se valora positivamente probar con distintas posiciones del hueco inicial e identificar aquellas en las que es plausible terminar con la última pieza en el propio hueco (problema *complementario* del peg solitaire). Aquí mostramos el estudio sobre la cruz inglesa clásica (V4) como tablero canónico.

Para el barrido completo sobre las cinco variantes (incluidas las obligatorias V1, V3 y V5), hay que ejecutar `python senku/scripts/experimento_huecos_completo.py`, que genera `senku/resultados/huecos_completo.csv` y `huecos_resumen.csv`.

In [ ]:
from senku.src.tableros import _tablero

base = TABLEROS[4]
candidatas = sorted({c for c in base.casillas if c[0] <= 3 and c[1] <= 3})
filas_huecos = []
for hueco in candidatas:
    t = _tablero(f'cruz_hueco_{hueco[0]}_{hueco[1]}', set(base.casillas), hueco=hueco, objetivo=hueco)
    p = ProblemaSenku.desde_tablero(t, modo_relajado=False)
    h = heuristica_conectividad(p)
    r = beam_search_con_reinicios(p, h, beta=500, intentos=8, iteraciones_maximas=120)
    filas_huecos.append({'hueco': hueco, 'complementario_ok': r.exito,
                         'movimientos': len(r.movimientos) if r.exito else None,
                         'nodos': r.nodos_expandidos,
                         'tiempo_s': round(r.tiempo_segundos, 1)})
pd.DataFrame(filas_huecos)

## 9. Conclusiones experimentales

1. **BFS no escala**. El espacio de estados crece exponencialmente y BFS no encuentra solución en las variantes medianas con un presupuesto razonable de nodos.
2. **Fast Downward es útil pero no universal**. Resuelve V3 (cruz asimétrica) en 8s y V4 (cruz inglesa) en 18s, pero se atasca en V1 (octógono), V2 (cruz griega grande) y V5 (rombo) incluso subiendo el timeout hasta 300s.
3. **Beam Search con pagoda no resuelve**. La pagoda es admisible pero no discrimina; ni con beta=2000 ni con 20 reinicios resuelve la cruz inglesa.
4. **La heurística de conectividad es la clave**. Al ordenar por número de componentes conexas, beam search resuelve las tres variantes obligatorias (1, 3 y 5).
5. **Beam search complementa a Fast Downward**. Las variantes donde FD se atasca (V1, V5) son justo aquellas en las que nuestro sistema sí encuentra plan, así que los dos enfoques se complementan.
6. **Estudio de huecos**. El barrido exhaustivo identifica qué huecos iniciales admiten plan en cada variante (resultados completos en los CSV de `senku/resultados/`).